# Phase 2 | GMM vs. Autoencoder Fraud Triage

Compare the saved full-covariance GMM (10 components) and standard autoencoder (12 latent dimensions) with a validation-selected blend. Larger anomaly scores rank first. The validation split determines score normalization, the blend weight, and the false-positive budgets. The chronological held-out test split measures ranking, queue yield, and fraud overlap.

Run `python -m src.preprocess` from the repository root to produce the required CSVs. This notebook uses the checked-in checkpoints and their original training scalers.

## Saved models and chronological validation split

Both estimators score the same 29 features using their own stored training scaler. The GMM uses negative log likelihood. The autoencoder uses mean squared reconstruction error and runs in CPU inference mode. Only load the repository's trusted model artifacts.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import StandardScaler

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.autoencoder import FraudAutoencoder
from src.preprocess import FEATURE_COLS, TARGET_COL

model_dir = root / "models"
processed = root / "data" / "processed"
gmm = joblib.load(model_dir / "best_gmm.joblib")
gmm_scaler = joblib.load(model_dir / "scaler.joblib")
ae_scaler = joblib.load(model_dir / "scaler_dedup.joblib")
checkpoint = torch.load(model_dir / "best_autoencoder.pt", map_location="cpu", weights_only=True)
ae = FraudAutoencoder(
    input_dim=len(FEATURE_COLS),
    hidden_dims=checkpoint["hidden_dims"],
    latent_dim=checkpoint["latent_dim"],
    dropout_rate=checkpoint["dropout_rate"],
)
ae.load_state_dict(checkpoint["state_dict"])
ae.eval()
assert gmm.n_components == 10 and gmm.covariance_type == "full"
assert ae.latent_dim == 12


def load_split(name, expected_rows, expected_frauds):
    path = processed / f"{name}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} is missing. Run python -m src.preprocess first.")
    frame = pd.read_csv(path, usecols=[*FEATURE_COLS, TARGET_COL])
    assert len(frame) == expected_rows, f"Unexpected size for {name}"
    assert frame[TARGET_COL].isin([0, 1]).all(), f"Unexpected classes in {name}"
    assert frame[TARGET_COL].sum() == expected_frauds, f"Unexpected fraud count for {name}"
    return frame


def base_scores(frame):
    features = frame[FEATURE_COLS]
    scores = pd.DataFrame({
        "GMM": -gmm.score_samples(gmm_scaler.transform(features)),
        "Autoencoder": ae.reconstruction_errors(ae_scaler.transform(features.to_numpy())),
    }, index=frame.index, dtype=float)
    assert np.isfinite(scores.to_numpy()).all()
    return scores


validation = load_split("val", expected_rows=56_961, expected_frauds=57)
y_val = validation[TARGET_COL].astype(bool)
val_scores = base_scores(validation)
print(f"Validation: {len(y_val):,} rows, {y_val.sum():,} frauds.")

## Validation-only blend selection

Fit a per-model z-score transform on all validation scores. Sweep the prespecified GMM weights and select by validation average precision (AP). If AP ties, prefer the weight closest to 0.50, then the smaller weight. Base models were trained on legitimate data. Blend selection uses validation labels.

In [ ]:
normalizer = StandardScaler().fit(val_scores)


def normalized(scores):
    return pd.DataFrame(normalizer.transform(scores), index=scores.index, columns=scores.columns)


def blend(z_scores, alpha):
    return alpha * z_scores["GMM"] + (1 - alpha) * z_scores["Autoencoder"]


val_z = normalized(val_scores)
sweep = []
for weight in (0.0, 0.25, 0.5, 0.75, 1.0):
    score = blend(val_z, weight)
    sweep.append((weight, average_precision_score(y_val, score), roc_auc_score(y_val, score)))

validation_ranking = pd.DataFrame(sweep, columns=["GMM weight", "Validation AP", "Validation ROC-AUC"])
alpha, _, _ = max(sweep, key=lambda result: (result[1], -abs(result[0] - 0.5), -result[0]))
val_scores["Hybrid"] = blend(val_z, alpha)

display(validation_ranking.round(6))
print(f"Locked GMM weight: {alpha:.2f}")

## False-positive budgets and locked cutoffs

With $N_0$ legitimate validation rows and target $b$ basis points, at most $\lfloor bN_0/10000\rfloor$ legitimate rows may be flagged. Sort their scores in descending order and use the first excluded score as the cutoff. Flag with a strict greater-than comparison. A tie at the boundary can leave capacity unused. The same score-space thresholds and comparison rule apply on test.

In [ ]:
budgets_bp = (10, 50, 100)
calibration = []
for model in val_scores:
    legit_desc = np.sort(val_scores.loc[~y_val, model].to_numpy())[::-1]
    for budget_bp in budgets_bp:
        allowed = budget_bp * len(legit_desc) // 10_000
        cutoff = legit_desc[allowed]
        val_fp = int(np.count_nonzero(legit_desc > cutoff))
        assert val_fp <= allowed
        calibration.append((model, budget_bp / 100, cutoff, val_fp, 100 * val_fp / len(legit_desc)))

cutoffs = pd.DataFrame(
    calibration,
    columns=["Model", "FPR target (%)", "Cutoff", "Val FP", "Val FPR (%)"],
)
display(cutoffs.round(6))